### 7.7 Prioritized Signal List

Select the top signals for manual literature review and DrugBank validation. Ranked by ROR within high-risk drug class pairs. Exported as `FAERS_DDI_PRIORITY_LIST.csv`.

## Prerequisites

**Mostly self-contained — loads signals from CSVs internally.**

This notebook covers Sections 7.7-7.10 of the original: prioritization, DrugBank validation, novel signal cleaning.

**Data required:**
- `../data/signals/FAERS_DDI_SIGNALS_HIGH_CONFIDENCE.csv` — HC signals (loaded internally)
- `../data/raw/drug_atc_mapping.csv` — ATC class mapping
- `/Users/joshbuck/Downloads/full_database_drugbank.xml` — DrugBank XML (for 7.8 parsing)
- `../data/raw/DRUGBANK_DDI_REFERENCE.csv` — parsed DrugBank (if skipping XML parsing)

**⚠️ DrugBank XML parsing (cell 7.8) takes 2-5 minutes and requires the DrugBank XML file.** If you already have `../data/raw/DRUGBANK_DDI_REFERENCE.csv`, you can skip that cell.

In [80]:
# =============================================================================
# 7.7 CREATE PRIORITIZED SIGNAL LIST FOR VALIDATION
# =============================================================================
# Top signals by ROR + high-risk drug classes


print("CREATING PRIORITIZED SIGNAL LIST")
print("=" * 60)

# Load high-confidence signals
signals = pd.read_csv("../data/signals/FAERS_DDI_SIGNALS_HIGH_CONFIDENCE.csv")
print(f"High-confidence signals loaded: {len(signals):,}")

# Use high-confidence signals for high-risk class filtering too
all_signals = pd.read_csv("../data/signals/FAERS_DDI_SIGNALS_HIGH_CONFIDENCE.csv")
print(f"High-confidence signals loaded for class filtering: {len(all_signals):,}")

# -----------------------------------------------------------------------------
# PART 1: Top 200 by ROR (from high-confidence signals)
# -----------------------------------------------------------------------------
print("\n" + "=" * 60)
print("PART 1: Top 200 signals by ROR magnitude")

top_by_ror = signals.nlargest(200, 'ROR')[['PAIR', 'N_EXPOSED', 'A_SERIOUS_EXPOSED', 'ROR', 'CI_LOWER', 'CI_UPPER']]
top_by_ror['PRIORITY_REASON'] = 'TOP_ROR'
print(f"Selected: {len(top_by_ror)} signals")
print(f"ROR range: {top_by_ror['ROR'].min():.1f} - {top_by_ror['ROR'].max():.1f}")

# -----------------------------------------------------------------------------
# PART 2: High-risk drug classes
# -----------------------------------------------------------------------------
print("\n" + "=" * 60)
print("PART 2: High-risk drug class signals")

high_risk_classes = {
    'ANTICOAGULANTS': ['WARFARIN', 'HEPARIN', 'ENOXAPARIN', 'RIVAROXABAN', 'APIXABAN', 'DABIGATRAN', 'EDOXABAN', 'FONDAPARINUX', 'COUMADIN', 'XARELTO', 'ELIQUIS', 'PRADAXA', 'LOVENOX'],
    'ANTIPLATELET': ['ASPIRIN', 'CLOPIDOGREL', 'PLAVIX', 'TICAGRELOR', 'PRASUGREL', 'BRILINTA', 'EFFIENT', 'DIPYRIDAMOLE'],
    'IMMUNOSUPPRESSANTS': ['TACROLIMUS', 'CYCLOSPORINE', 'MYCOPHENOLATE', 'AZATHIOPRINE', 'SIROLIMUS', 'EVEROLIMUS', 'PROGRAF', 'CELLCEPT', 'IMURAN'],
    'BIOLOGICS': ['ADALIMUMAB', 'INFLIXIMAB', 'ETANERCEPT', 'RITUXIMAB', 'TOCILIZUMAB', 'HUMIRA', 'REMICADE', 'ENBREL', 'ACTEMRA', 'DUPIXENT', 'VEDOLIZUMAB'],
    'OPIOIDS': ['OXYCODONE', 'HYDROCODONE', 'MORPHINE', 'FENTANYL', 'TRAMADOL', 'CODEINE', 'METHADONE', 'BUPRENORPHINE', 'HYDROMORPHONE'],
    'ANTINEOPLASTICS': ['METHOTREXATE', 'CYCLOPHOSPHAMIDE', 'DOXORUBICIN', 'PACLITAXEL', 'CARBOPLATIN', 'CISPLATIN', 'FLUOROURACIL', 'VINCRISTINE', 'IMATINIB', 'PEMBROLIZUMAB', 'NIVOLUMAB'],
    'NARROW_THERAPEUTIC_INDEX': ['DIGOXIN', 'LITHIUM', 'PHENYTOIN', 'CARBAMAZEPINE', 'THEOPHYLLINE', 'VALPROIC', 'AMINOGLYCOSIDE', 'GENTAMICIN', 'VANCOMYCIN']
}

def get_risk_classes(pair):
    pair_upper = pair.upper()
    classes_found = []
    for class_name, drugs in high_risk_classes.items():
        for drug in drugs:
            if drug in pair_upper:
                classes_found.append(class_name)
                break
    return classes_found

all_signals['RISK_CLASSES'] = all_signals['PAIR'].apply(get_risk_classes)
all_signals['NUM_RISK_CLASSES'] = all_signals['RISK_CLASSES'].apply(len)
all_signals['HAS_HIGH_RISK'] = all_signals['NUM_RISK_CLASSES'] > 0

high_risk_signals = all_signals[all_signals['HAS_HIGH_RISK']].copy()
print(f"Signals involving high-risk drugs: {len(high_risk_signals):,}")

print("\nSignals by drug class:")
for class_name in high_risk_classes.keys():
    count = high_risk_signals['RISK_CLASSES'].apply(lambda x: class_name in x).sum()
    print(f"  {class_name}: {count:,}")

high_risk_top = high_risk_signals[~high_risk_signals['PAIR'].isin(top_by_ror['PAIR'])]
high_risk_top = high_risk_top.nlargest(100, 'ROR')[['PAIR', 'N_EXPOSED', 'A_SERIOUS_EXPOSED', 'ROR', 'CI_LOWER', 'CI_UPPER', 'RISK_CLASSES']]
high_risk_top['PRIORITY_REASON'] = 'HIGH_RISK_CLASS'
print(f"\nAdditional high-risk signals (not in top ROR): {len(high_risk_top)}")

# -----------------------------------------------------------------------------
# PART 3: Combine into priority list
# -----------------------------------------------------------------------------
print("\n" + "=" * 60)
print("PART 3: Combined priority list")

priority_list = pd.concat([
    top_by_ror,
    high_risk_top[['PAIR', 'N_EXPOSED', 'A_SERIOUS_EXPOSED', 'ROR', 'CI_LOWER', 'CI_UPPER', 'PRIORITY_REASON']]
], ignore_index=True)

priority_list = priority_list.drop_duplicates(subset='PAIR', keep='first')
priority_list['RISK_CLASSES'] = priority_list['PAIR'].apply(get_risk_classes)
priority_list['RISK_CLASS_STR'] = priority_list['RISK_CLASSES'].apply(lambda x: ', '.join(x) if x else 'NONE')
priority_list = priority_list.sort_values('ROR', ascending=False)

print(f"Total priority signals: {len(priority_list):,}")
print(f"  - From top ROR: {(priority_list['PRIORITY_REASON'] == 'TOP_ROR').sum()}")
print(f"  - From high-risk classes: {(priority_list['PRIORITY_REASON'] == 'HIGH_RISK_CLASS').sum()}")

priority_list.to_csv("../data/signals/FAERS_DDI_PRIORITY_LIST.csv", index=False)
print(f"\n Saved: FAERS_DDI_PRIORITY_LIST.csv")

print("\n" + "=" * 60)
print("TOP 30 PRIORITY SIGNALS FOR VALIDATION")
print("=" * 60)
display_cols = ['PAIR', 'N_EXPOSED', 'ROR', 'CI_LOWER', 'RISK_CLASS_STR', 'PRIORITY_REASON']
print(priority_list[display_cols].head(30).to_string())

CREATING PRIORITIZED SIGNAL LIST
High-confidence signals loaded: 5,472
High-confidence signals loaded for class filtering: 5,472

PART 1: Top 200 signals by ROR magnitude
Selected: 200 signals
ROR range: 6.9 - 8.7

PART 2: High-risk drug class signals
Signals involving high-risk drugs: 1,458

Signals by drug class:
  ANTICOAGULANTS: 252
  ANTIPLATELET: 161
  IMMUNOSUPPRESSANTS: 103
  BIOLOGICS: 349
  OPIOIDS: 319
  ANTINEOPLASTICS: 222
  NARROW_THERAPEUTIC_INDEX: 115

Additional high-risk signals (not in top ROR): 100

PART 3: Combined priority list
Total priority signals: 300
  - From top ROR: 200
  - From high-risk classes: 100

 Saved: FAERS_DDI_PRIORITY_LIST.csv

TOP 30 PRIORITY SIGNALS FOR VALIDATION
                                                                     PAIR  N_EXPOSED       ROR  CI_LOWER      RISK_CLASS_STR PRIORITY_REASON
0      CETIRIZINE HYDROCHLORIDE\PSEUDOEPHEDRINE HYDROCHLORIDE + RITUXIMAB        170  8.702641  5.272603           BIOLOGICS         TOP_ROR
1  

### 7.8 Parse DrugBank Reference Database

Extract all known drug-drug interactions from the DrugBank full XML export (licensed data, not redistributed). Builds a lookup set of ~1.4M known interaction pairs used in Section 7.9 validation.

In [81]:
# =============================================================================
# 7.8 PARSE DRUGBANK XML FOR DRUG-DRUG INTERACTIONS
# =============================================================================
# Extract known DDIs from DrugBank database



print("PARSING DRUGBANK XML")
print("=" * 60)

# DrugBank XML file path
DRUGBANK_XML = "/Users/joshbuck/Downloads/full_database_drugbank.xml"

# Check if file exists
if not os.path.exists(DRUGBANK_XML):
    print(f"ERROR: DrugBank XML not found at {DRUGBANK_XML}")
    print("Please check the filename and path")
else:
    print(f"Parsing: {DRUGBANK_XML}")
    print("This may take 2-5 minutes for the full database...")

    # Parse XML incrementally (memory efficient)
    drugbank_ddis = []
    drug_names = {}  # drugbank_id -> name
    drug_synonyms = defaultdict(set)  # name -> all synonyms

    # DrugBank namespace
    ns = {'db': 'http://www.drugbank.ca'}

    # Iterate through drugs
    context = ET.iterparse(DRUGBANK_XML, events=('end',))

    drug_count = 0
    ddi_count = 0

    for event, elem in context:
        if elem.tag == '{http://www.drugbank.ca}drug':
            drug_count += 1

            # Get drug ID and name
            drugbank_id = elem.find('db:drugbank-id[@primary="true"]', ns)
            name_elem = elem.find('db:name', ns)

            if drugbank_id is not None and name_elem is not None:
                db_id = drugbank_id.text
                name = name_elem.text.upper() if name_elem.text else None

                if name:
                    drug_names[db_id] = name
                    drug_synonyms[name].add(name)

                    # Get synonyms
                    synonyms = elem.findall('.//db:synonym', ns)
                    for syn in synonyms:
                        if syn.text:
                            drug_synonyms[name].add(syn.text.upper())

                    # Get brand names
                    brands = elem.findall('.//db:product/db:name', ns)
                    for brand in brands:
                        if brand.text:
                            drug_synonyms[name].add(brand.text.upper())

                # Get drug interactions
                interactions = elem.findall('.//db:drug-interaction', ns)
                for interaction in interactions:
                    target_id = interaction.find('db:drugbank-id', ns)
                    target_name = interaction.find('db:name', ns)
                    description = interaction.find('db:description', ns)

                    if target_name is not None and target_name.text:
                        drugbank_ddis.append({
                            'DRUG_A': name,
                            'DRUG_B': target_name.text.upper(),
                            'DESCRIPTION': description.text if description is not None else None,
                            'SOURCE': 'DrugBank'
                        })
                        ddi_count += 1

            # Clear element to save memory
            elem.clear()

            # Progress update
            if drug_count % 10000 == 0:
                print(f"  Processed {drug_count:,} drugs, {ddi_count:,} DDIs found...")

    print(f"\nParsing complete!")
    print(f"  Total drugs: {drug_count:,}")
    print(f"  Total DDIs extracted: {ddi_count:,}")

    # Create DataFrame
    drugbank_df = pd.DataFrame(drugbank_ddis)

    # Create standardized pair names (alphabetical order)
    drugbank_df['PAIR'] = drugbank_df.apply(
        lambda row: ' + '.join(sorted([row['DRUG_A'], row['DRUG_B']])),
        axis=1
    )

    # Remove duplicates (A-B and B-A are same interaction)
    drugbank_df = drugbank_df.drop_duplicates(subset='PAIR')

    print(f"  Unique DDI pairs: {len(drugbank_df):,}")

    # Save for reuse
    drugbank_df.to_csv("../data/raw/DRUGBANK_DDI_REFERENCE.csv", index=False)
    print(f"\n Saved: DRUGBANK_DDI_REFERENCE.csv")

    # Create lookup set for fast matching
    drugbank_pairs = set(drugbank_df['PAIR'].tolist())
    print(f"DrugBank DDI lookup ready: {len(drugbank_pairs):,} known interactions")

PARSING DRUGBANK XML
Parsing: /Users/joshbuck/Downloads/full_database_drugbank.xml
This may take 2-5 minutes for the full database...
  Processed 10,000 drugs, 54,324 DDIs found...
  Processed 20,000 drugs, 54,847 DDIs found...
  Processed 30,000 drugs, 55,072 DDIs found...
  Processed 40,000 drugs, 55,074 DDIs found...
  Processed 50,000 drugs, 55,681 DDIs found...
  Processed 60,000 drugs, 55,681 DDIs found...
  Processed 70,000 drugs, 56,898 DDIs found...
  Processed 80,000 drugs, 56,898 DDIs found...
  Processed 90,000 drugs, 56,898 DDIs found...
  Processed 100,000 drugs, 56,898 DDIs found...
  Processed 110,000 drugs, 56,898 DDIs found...
  Processed 120,000 drugs, 56,898 DDIs found...
  Processed 130,000 drugs, 56,898 DDIs found...
  Processed 140,000 drugs, 56,898 DDIs found...
  Processed 150,000 drugs, 56,898 DDIs found...
  Processed 160,000 drugs, 56,898 DDIs found...
  Processed 170,000 drugs, 56,898 DDIs found...
  Processed 180,000 drugs, 56,898 DDIs found...
  Processed

### 7.9 Validate FAERS Signals Against DrugBank

Cross-reference our ROR signals against the DrugBank known-DDI set. Drug name matching uses salt-stripping (e.g., removes ' HYDROCHLORIDE', ' SODIUM') consistent with FAERS naming conventions. Signals are classified as **Known** (DrugBank-confirmed) or **Novel** (not in DrugBank).

In [82]:
# =============================================================================
# 7.9 VALIDATE FAERS SIGNALS AGAINST DRUGBANK
# =============================================================================
# Cross-reference our detected signals with known DDIs

print("VALIDATING FAERS SIGNALS AGAINST DRUGBANK")
print("=" * 60)

# Load our priority signals
priority_signals = pd.read_csv("../data/signals/FAERS_DDI_PRIORITY_LIST.csv")
print(f"Priority signals to validate: {len(priority_signals):,}")

# Load DrugBank DDIs
drugbank_df = pd.read_csv("../data/raw/DRUGBANK_DDI_REFERENCE.csv")
print(f"DrugBank known DDIs: {len(drugbank_df):,}")

# Create lookup set of DrugBank pairs (both orderings)
drugbank_pairs = set()
drugbank_drugs = set()

for _, row in drugbank_df.iterrows():
    a, b = row['DRUG_A'], row['DRUG_B']
    drugbank_pairs.add(f"{a} + {b}")
    drugbank_pairs.add(f"{b} + {a}")
    drugbank_drugs.add(a)
    drugbank_drugs.add(b)

print(f"DrugBank lookup set: {len(drugbank_pairs):,} pairs (both orderings)")
print(f"Unique drugs in DrugBank: {len(drugbank_drugs):,}")

# -----------------------------------------------------------------------------
# Exact matching (fast - just set lookup)
# -----------------------------------------------------------------------------
print("\n" + "=" * 60)
print("EXACT MATCHING")



def check_drugbank_match(pair):
    """
    Check if pair exists in DrugBank.
    For combination products (containing / or \\), check if ANY component
    of one side interacts with ANY component of the other side.
    """
    pair_upper = pair.upper()

    # First try exact match
    if pair_upper in drugbank_pairs:
        return True

    # Split the pair into two sides
    sides = pair_upper.split(' + ')
    if len(sides) != 2:
        return False

    # Split each side into individual components
    import re
    components_a = [c.strip() for c in re.split(r'[/\\]', sides[0]) if len(c.strip()) > 2]
    components_b = [c.strip() for c in re.split(r'[/\\]', sides[1]) if len(c.strip()) > 2]

    # Remove salt forms and suffixes for matching
    def clean_for_matching(name):
        for suffix in [' HYDROCHLORIDE', ' SODIUM', ' PHOSPHATE', ' SULFATE',
                       ' POTASSIUM', ' BESYLATE', ' MALEATE', ' FUMARATE',
                       ' MESYLATE', ' ACETATE', ' DISODIUM', ' CALCIUM']:
            name = name.replace(suffix, '')
        return name.strip()

    components_a = [clean_for_matching(c) for c in components_a]
    components_b = [clean_for_matching(c) for c in components_b]

    # Check if any component from side A interacts with any component from side B
    for comp_a in components_a:
        for comp_b in components_b:
            check1 = f"{comp_a} + {comp_b}"
            check2 = f"{comp_b} + {comp_a}"
            if check1 in drugbank_pairs or check2 in drugbank_pairs:
                return True

    return False

priority_signals['IN_DRUGBANK'] = priority_signals['PAIR'].apply(check_drugbank_match)
matches = priority_signals['IN_DRUGBANK'].sum()
print(f"DrugBank matches found: {matches} / {len(priority_signals)} ({100*matches/len(priority_signals):.1f}%)")


# -----------------------------------------------------------------------------
# Fuzzy matching - check if individual drugs exist in DrugBank
# -----------------------------------------------------------------------------
print("\n" + "=" * 60)
print("COMPONENT DRUG MATCHING")


def check_drugs_in_drugbank(pair):
    """Check if each drug in the pair exists in DrugBank"""
    import re
    drugs = pair.upper().split(' + ')
    if len(drugs) != 2:
        return 'NEITHER_KNOWN'

    def drug_in_drugbank(drug_name):
        """Check if a drug (or any component of a combination) is in DrugBank"""
        # Split combinations into components
        components = [c.strip() for c in re.split(r'[/\\]', drug_name) if len(c.strip()) > 2]

        # Strip salt forms
        def clean(name):
            for suffix in [' HYDROCHLORIDE', ' SODIUM', ' PHOSPHATE', ' SULFATE',
                           ' POTASSIUM', ' BESYLATE', ' MALEATE', ' FUMARATE',
                           ' MESYLATE', ' ACETATE', ' DISODIUM', ' CALCIUM']:
                name = name.replace(suffix, '')
            return name.strip()

        components = [clean(c) for c in components]

        for comp in components:
            # Exact match only — no substring matching
            if comp in drugbank_drugs:
                return True
        return False

    a_found = drug_in_drugbank(drugs[0].strip())
    b_found = drug_in_drugbank(drugs[1].strip())

    if a_found and b_found:
        return 'BOTH_DRUGS_KNOWN'
    elif a_found or b_found:
        return 'ONE_DRUG_KNOWN'
    else:
        return 'NEITHER_KNOWN'



priority_signals['DRUG_STATUS'] = priority_signals['PAIR'].apply(check_drugs_in_drugbank)
print(priority_signals['DRUG_STATUS'].value_counts())

# -----------------------------------------------------------------------------
# Categorize signals
# -----------------------------------------------------------------------------
print("\n" + "=" * 60)
print("SIGNAL CATEGORIZATION")

def categorize_signal(row):
    if row['IN_DRUGBANK']:
        return 'KNOWN_DDI'
    elif row['DRUG_STATUS'] == 'BOTH_DRUGS_KNOWN':
        return 'POTENTIALLY_NOVEL'
    elif row['DRUG_STATUS'] == 'ONE_DRUG_KNOWN':
        return 'PARTIAL_MATCH'
    else:
        return 'UNKNOWN_DRUGS'

priority_signals['VALIDATION_STATUS'] = priority_signals.apply(categorize_signal, axis=1)

print(f"\nValidation Results:")
print(priority_signals['VALIDATION_STATUS'].value_counts())

# -----------------------------------------------------------------------------
# Get DrugBank descriptions for known DDIs
# -----------------------------------------------------------------------------
print("\n" + "=" * 60)
print("KNOWN DDIs (validates our method)")
print("=" * 60)

known = priority_signals[priority_signals['VALIDATION_STATUS'] == 'KNOWN_DDI'].copy()
print(f"Found {len(known)} signals that match known DrugBank DDIs")

# Add DrugBank descriptions
def get_drugbank_description(pair):
    pair_upper = pair.upper()
    drugs = pair_upper.split(' + ')
    # Try both orderings
    match = drugbank_df[(drugbank_df['DRUG_A'] == drugs[0]) & (drugbank_df['DRUG_B'] == drugs[1])]
    if len(match) == 0:
        match = drugbank_df[(drugbank_df['DRUG_A'] == drugs[1]) & (drugbank_df['DRUG_B'] == drugs[0])]
    if len(match) > 0:
        return match.iloc[0]['DESCRIPTION']
    return None

if len(known) > 0:
    known['DRUGBANK_DESCRIPTION'] = known['PAIR'].apply(get_drugbank_description)
    print("\nTop 15 Known DDIs with DrugBank Descriptions:")
    for idx, row in known.head(15).iterrows():
        desc = row['DRUGBANK_DESCRIPTION']
        desc_short = desc[:100] + "..." if desc and len(desc) > 100 else desc
        print(f"\n{row['PAIR']} (ROR={row['ROR']:.1f}, N={row['N_EXPOSED']})")
        print(f"  DrugBank: {desc_short}")

# -----------------------------------------------------------------------------
# Novel signals
# -----------------------------------------------------------------------------
print("\n" + "=" * 60)
print("POTENTIALLY NOVEL SIGNALS (for further investigation)")
print("=" * 60)

novel = priority_signals[priority_signals['VALIDATION_STATUS'] == 'POTENTIALLY_NOVEL'].copy()
print(f"Found {len(novel)} signals where both drugs are in DrugBank but pair is NOT a known DDI")
print("These are candidates for novel DDI discovery!")

if len(novel) > 0:
    novel_sorted = novel.sort_values('N_EXPOSED', ascending=False)
    print("\nTop 20 Novel Signals (by sample size):")
    print(novel_sorted[['PAIR', 'N_EXPOSED', 'ROR', 'RISK_CLASS_STR']].head(20).to_string())

# -----------------------------------------------------------------------------
# Save results
# -----------------------------------------------------------------------------
priority_signals.to_csv("../data/validated/FAERS_DDI_VALIDATED.csv", index=False)
print(f"\n Saved: FAERS_DDI_VALIDATED.csv")

known.to_csv("../data/validated/FAERS_DDI_KNOWN.csv", index=False)
print(f" Saved: FAERS_DDI_KNOWN.csv ({len(known)} signals)")

novel.to_csv("../data/validated/FAERS_DDI_NOVEL.csv", index=False)
print(f" Saved: FAERS_DDI_NOVEL.csv ({len(novel)} signals)")

# -----------------------------------------------------------------------------
# Final summary
# -----------------------------------------------------------------------------
print(f"\n{'='*60}")
print("VALIDATION SUMMARY")
print(f"{'='*60}")
print(f"Total priority signals validated: {len(priority_signals):,}")
print(f"   Known DDIs (exact match in DrugBank): {len(known)}")
print(f"   Potentially novel (both drugs known, pair not): {len(novel)}")
print(f"   Partial match (one drug known): {(priority_signals['VALIDATION_STATUS'] == 'PARTIAL_MATCH').sum()}")
print(f"   Unknown drugs: {(priority_signals['VALIDATION_STATUS'] == 'UNKNOWN_DRUGS').sum()}")

if len(known) > 0:
    validation_rate = 100 * len(known) / len(priority_signals)
    print(f"\nMethod Validation: {validation_rate:.1f}% of top signals are confirmed known DDIs")
    print("This demonstrates our FAERS-based detection method is working correctly")

VALIDATING FAERS SIGNALS AGAINST DRUGBANK
Priority signals to validate: 300
DrugBank known DDIs: 1,455,276
DrugBank lookup set: 2,910,552 pairs (both orderings)
Unique drugs in DrugBank: 4,629

EXACT MATCHING
DrugBank matches found: 81 / 300 (27.0%)

COMPONENT DRUG MATCHING
DRUG_STATUS
BOTH_DRUGS_KNOWN    150
ONE_DRUG_KNOWN      126
NEITHER_KNOWN        24
Name: count, dtype: int64

SIGNAL CATEGORIZATION

Validation Results:
VALIDATION_STATUS
PARTIAL_MATCH        126
KNOWN_DDI             81
POTENTIALLY_NOVEL     69
UNKNOWN_DRUGS         24
Name: count, dtype: int64

KNOWN DDIs (validates our method)
Found 81 signals that match known DrugBank DDIs

Top 15 Known DDIs with DrugBank Descriptions:

CELECOXIB + SULFASALAZINE (ROR=8.7, N=249)
  DrugBank: The risk or severity of adverse effects can be increased when Celecoxib is combined with Sulfasalazi...

KETAMINE + OLANZAPINE (ROR=8.6, N=109)
  DrugBank: The risk or severity of CNS depression can be increased when Olanzapine is combined w

### 7.9.1 Novel Signal Cleaning

Remove artefactual signals from the novel list: same-drug pairs, known interactions missed by name-matching, and low-plausibility combinations. Exports cleaned novel signals to `FAERS_DDI_NOVEL_CLEANED.csv`.

In [83]:
# =============================================================================
# 7.9.1 NOVEL SIGNAL CLEANING
# =============================================================================
# Remove duplicate/same-drug artifacts and reclassify known interactions
# that DrugBank missed due to name mismatches

import re

print("NOVEL SIGNAL CLEANING")
print("=" * 60)

novel = pd.read_csv("../data/validated/FAERS_DDI_NOVEL.csv")
print(f"Original novel signals: {len(novel):,}")

# -----------------------------------------------------------------
# FILTER 1: Remove same-drug / combination product artifacts
# -----------------------------------------------------------------
# If DRUG_A is a component of DRUG_B (or vice versa), it's not a real pair

def is_same_drug(pair):
    """Detect when both sides of a pair contain the same active ingredient"""
    drugs = pair.upper().split(' + ')
    if len(drugs) != 2:
        return True

    a = drugs[0].strip()
    b = drugs[1].strip()

    # Extract individual ingredient words
    def get_ingredients(name):
        # Remove dosage info like "160 MG", "IN 5 ML", etc.
        name = re.sub(r'\d+\s*(MG|ML|MCG|G|MG/ML|%)\b', '', name)
        # Remove formulation words
        for word in ['ORAL', 'SUSPENSION', 'TABLET', 'CAPSULE', 'PILL', 'INJECTION',
                     'HYDROCHLORIDE', 'SODIUM', 'PHOSPHATE', 'SULFATE', 'POTASSIUM',
                     'FOR', 'CHILDRENS', 'IN', 'NOVALGINA', 'ACETAMINOPHEN PAIN AND FEVER']:
            name = name.replace(word, '')
        # Split on separators
        parts = re.split(r'[/\\,+]', name)
        # Clean each part
        ingredients = set()
        for p in parts:
            p = p.strip()
            if len(p) > 2:  # Skip empty or very short fragments
                ingredients.add(p)
        return ingredients

    ingredients_a = get_ingredients(a)
    ingredients_b = get_ingredients(b)

    # Check if any ingredient appears in both sides
    overlap = ingredients_a & ingredients_b
    if overlap:
        return True

    # Check if one name is contained in the other
    # e.g., "ACETAMINOPHEN" in "ACETAMINOPHEN / CODEINE"
    for ing_a in ingredients_a:
        if ing_a in b and len(ing_a) > 4:
            return True
    for ing_b in ingredients_b:
        if ing_b in a and len(ing_b) > 4:
            return True

    return False

novel['IS_SAME_DRUG'] = novel['PAIR'].apply(is_same_drug)
same_drug_count = novel['IS_SAME_DRUG'].sum()
print(f"\nFilter 1 - Same-drug artifacts found: {same_drug_count}")
if same_drug_count > 0:
    print("Examples removed:")
    for pair in novel[novel['IS_SAME_DRUG']]['PAIR'].head(10).tolist():
        print(f"  REMOVED{pair}")

# -----------------------------------------------------------------
# FILTER 2: Reclassify known interactions missed by name mismatch
# -----------------------------------------------------------------
# These are well-documented DDIs that DrugBank exact matching missed
# because FAERS uses international spellings or salt forms

known_interactions = [
    # ARB/ACE + NSAID (class-level interaction, well documented)
    ('CANDESARTAN', 'DICLOFENAC'),  # DICLOFENACO in FAERS
    ('CANDESARTAN', 'CELECOXIB'),
    ('CANDESARTAN', 'IBUPROFEN'),
    # Acetaminophen + Alcohol (textbook hepatotoxicity)
    ('ACETAMINOPHEN', 'ALCOHOL'),
    # TNF inhibitor + immunosuppressant (FDA boxed warning)
    ('ADALIMUMAB', 'AZATHIOPRINE'),
    ('ADALIMUMAB', 'MERCAPTOPURINE'),
    ('INFLIXIMAB', 'AZATHIOPRINE'),
    # Opioid combinations
    ('OXYCODONE', 'CODEINE'),
    ('HYDROCODONE', 'CODEINE'),
]

def is_known_missed(pair):
    """Check if pair matches a known interaction that DrugBank missed"""
    pair_upper = pair.upper()
    for drug_a, drug_b in known_interactions:
        if drug_a in pair_upper and drug_b in pair_upper:
            return True
    return False

novel['IS_KNOWN_MISSED'] = novel['PAIR'].apply(is_known_missed)
known_missed_count = novel['IS_KNOWN_MISSED'].sum()
print(f"\nFilter 2 - Known interactions (name mismatch): {known_missed_count}")
if known_missed_count > 0:
    print("Reclassified as KNOWN (not novel):")
    for pair in novel[novel['IS_KNOWN_MISSED']]['PAIR'].head(10).tolist():
        print(f"  KNOWN {pair}")

# -----------------------------------------------------------------
# FILTER 3: Remove pairs with international drug name duplicates
# -----------------------------------------------------------------
# FAERS contains multiple spellings of the same drug names that are
# essentially the same drug with a different suffix

international_duplicates = {
    'DICLOFENACO': 'DICLOFENAC',
    'AMLODIPINO': 'AMLODIPINE',
    'ATORVASTATINA': 'ATORVASTATIN',
    'IBUPROFENO': 'IBUPROFEN',
    'OMEPRAZOL': 'OMEPRAZOLE',
    'PARACETAMOL': 'ACETAMINOPHEN',
    'METAMIZOL': 'METAMIZOLE',
    'DEXAMETASONA': 'DEXAMETHASONE',
    'PREDNISONA': 'PREDNISONE',
    'METOTREXATO': 'METHOTREXATE',
    'CICLOSPORINA': 'CYCLOSPORINE',
    'NALOXONA': 'NALOXONE',
    'DOXORUBICINA': 'DOXORUBICIN',
    'CETIRIZINA': 'CETIRIZINE',
    'ACIDO ALENDRONICO': 'ALENDRONATE',
    'CALCIO': 'CALCIUM',
}

def has_international_variant(pair):
    """Flag pairs where international spelling might cause false novelty"""
    pair_upper = pair.upper()
    for foreign, english in international_duplicates.items():
        if foreign in pair_upper:
            return True
    return False

novel['HAS_INTL_VARIANT'] = novel['PAIR'].apply(has_international_variant)
intl_count = novel['HAS_INTL_VARIANT'].sum()
print(f"\nFilter 3 - International name variants flagged: {intl_count}")
if intl_count > 0:
    print("Examples flagged for review:")
    for pair in novel[novel['HAS_INTL_VARIANT']]['PAIR'].head(10).tolist():
        print(f"  REVIEW  {pair}")


# -----------------------------------------------------------------
# FILTER 4: Manual exclusions — known artifacts
# -----------------------------------------------------------------
# Signals that aren't real DDIs due to abuse patterns, product naming
# issues, or other data quality problems in FAERS

MANUAL_EXCLUSIONS = [
    # Same drug — long product names that bypass ingredient overlap detection
    'ACETAMINOPHEN 160 MG',
    'NOVALGINA',

    # Topical-only drugs unlikely to cause systemic DDIs
    'DESOXIMETASONE',           # Topical corticosteroid

    # Rarely used / suspicious frequency in signals
    'PHTHALYLSULFATHIAZOLE',    # Old sulfonamide, rarely prescribed

    # Polydrug abuse markers — not drug interactions, just co-ingestion
    'BENZOYLECGONINE',      # Cocaine metabolite
    'ETHANOL',

    # Animal products / supplements that aren't real drugs
    'MARVEL AID',
    'CAVIAR',

    # Dialysis solutions — not systemic drugs
    'DELFLEX',
    'DIANEAL',
    'EXTRANEAL',

    # Experimental / unidentifiable compounds
    'BMS-830216',
    'REGUNEAL',
    'LIPOGEN',
]

def is_manual_exclusion(pair):
    """Flag pairs containing known non-drug or artifact entries"""
    pair_upper = pair.upper()
    for term in MANUAL_EXCLUSIONS:
        if term in pair_upper:
            return True
    return False

novel['IS_MANUAL_EXCLUSION'] = novel['PAIR'].apply(is_manual_exclusion)
manual_excl_count = novel['IS_MANUAL_EXCLUSION'].sum()
print(f"\nFilter 4 - Manual exclusions: {manual_excl_count}")
if manual_excl_count > 0:
    print("Removed:")
    for pair in novel[novel['IS_MANUAL_EXCLUSION']]['PAIR'].tolist():
        print(f"  EXCLUDED  {pair}")


# -----------------------------------------------------------------
# APPLY FILTERS
# -----------------------------------------------------------------
print("\n" + "=" * 60)
print("APPLYING FILTERS")

# Remove same-drug artifacts entirely
# Remove known-but-mismatched (reclassify)
# Flag international variants for manual review

novel_clean = novel[
    (~novel['IS_SAME_DRUG']) &
    (~novel['IS_KNOWN_MISSED']) &
    (~novel['IS_MANUAL_EXCLUSION'])
].copy()

# Mark international variants as needing review
novel_clean['NEEDS_REVIEW'] = novel_clean['HAS_INTL_VARIANT']

print(f"\nOriginal novel signals:     {len(novel):,}")
print(f"Removed (same-drug):        {same_drug_count}")
print(f"Removed (known missed):     {known_missed_count}")
print(f"Removed (manual exclusion): {manual_excl_count}")
print(f"Remaining novel signals:    {len(novel_clean):,}")
print(f"  - Clean (ready for lit search):  {(~novel_clean['NEEDS_REVIEW']).sum()}")
print(f"  - Flagged for review:            {novel_clean['NEEDS_REVIEW'].sum()}")

# -----------------------------------------------------------------
# SAVE CLEANED FILES
# -----------------------------------------------------------------

# Save cleaned novel signals
novel_clean.to_csv("../data/validated/FAERS_DDI_NOVEL_CLEANED.csv", index=False)
print(f"\n Saved: FAERS_DDI_NOVEL_CLEANED.csv ({len(novel_clean)} signals)")

# Save the reclassified known interactions separately
known_reclassified = novel[novel['IS_KNOWN_MISSED']].copy()
known_reclassified['RECLASSIFICATION_REASON'] = 'Name mismatch in DrugBank matching'
known_reclassified.to_csv("../data/validated/FAERS_DDI_KNOWN_RECLASSIFIED.csv", index=False)
print(f" Saved: FAERS_DDI_KNOWN_RECLASSIFIED.csv ({len(known_reclassified)} signals)")

# Save same-drug artifacts for documentation
same_drug_artifacts = novel[novel['IS_SAME_DRUG']].copy()
same_drug_artifacts.to_csv("../data/validated/FAERS_DDI_SAME_DRUG_ARTIFACTS.csv", index=False)
print(f" Saved: FAERS_DDI_SAME_DRUG_ARTIFACTS.csv ({len(same_drug_artifacts)} artifacts)")

# -----------------------------------------------------------------
# TOP 20 CLEANED NOVEL SIGNALS FOR LIT SEARCH
# -----------------------------------------------------------------
print("\n" + "=" * 60)
print("TOP 20 CLEANED NOVEL SIGNALS (by sample size)")
print("=" * 60)

lit_search_targets = novel_clean[~novel_clean['NEEDS_REVIEW']].nlargest(20, 'N_EXPOSED')
display_cols = ['PAIR', 'N_EXPOSED', 'ROR', 'CI_LOWER']
print(lit_search_targets[display_cols].to_string(index=False))

lit_search_targets.to_csv("../data/validated/FAERS_DDI_LIT_SEARCH_TARGETS.csv", index=False)
print(f"\n Saved: FAERS_DDI_LIT_SEARCH_TARGETS.csv")

# Filter lit search targets to plausible ROR range
lit_targets = pd.read_csv("../data/validated/FAERS_DDI_LIT_SEARCH_TARGETS.csv")

# Remove extreme ROR (likely confounded) and remaining duplicates
lit_targets_filtered = lit_targets[lit_targets['ROR'] < 50].copy()

# Also remove calcium carbonate + calcium citrate (same mineral)
lit_targets_filtered = lit_targets_filtered[
    ~lit_targets_filtered['PAIR'].str.contains('CALCIUM CARBONATE.*CALCIUM CITRATE', case=False)
]

print(f"Before filtering: {len(lit_targets)} signals")
print(f"After removing ROR > 50 and duplicates: {len(lit_targets_filtered)}")
print("\nLIT SEARCH TARGETS (plausible ROR range):")
print(lit_targets_filtered[['PAIR', 'N_EXPOSED', 'ROR', 'CI_LOWER']].to_string(index=False))

lit_targets_filtered.to_csv("../data/validated/FAERS_DDI_LIT_SEARCH_TARGETS.csv", index=False)
print("\nSaved updated search targets")

# Expand from the full cleaned novel signals
full_clean = pd.read_csv("../data/validated/FAERS_DDI_NOVEL_CLEANED.csv")
full_clean = full_clean[
    (full_clean['ROR'] < 50) &
    (~full_clean['NEEDS_REVIEW']) &
    (~full_clean['PAIR'].str.contains('CALCIUM CARBONATE.*CALCIUM CITRATE', case=False))
]

# Sort by N_EXPOSED and take top 25
expanded_targets = full_clean.nlargest(25, 'N_EXPOSED')
print(f"Expanded lit search targets: {len(expanded_targets)}")
print(expanded_targets[['PAIR', 'N_EXPOSED', 'ROR', 'CI_LOWER']].to_string(index=False))

expanded_targets.to_csv("../data/validated/FAERS_DDI_LIT_SEARCH_TARGETS.csv", index=False)
print("\nSaved expanded search targets")

NOVEL SIGNAL CLEANING
Original novel signals: 69

Filter 1 - Same-drug artifacts found: 0

Filter 2 - Known interactions (name mismatch): 0

Filter 3 - International name variants flagged: 0

Filter 4 - Manual exclusions: 8
Removed:
  EXCLUDED  CETIRIZINE HYDROCHLORIDE + DESOXIMETASONE
  EXCLUDED  DESOXIMETASONE + PHTHALYLSULFATHIAZOLE
  EXCLUDED  CETIRIZINE HYDROCHLORIDE\PSEUDOEPHEDRINE HYDROCHLORIDE + PHTHALYLSULFATHIAZOLE
  EXCLUDED  LEFLUNOMIDE + PHTHALYLSULFATHIAZOLE
  EXCLUDED  PHTHALYLSULFATHIAZOLE + SULFASALAZINE
  EXCLUDED  ALENDRONIC ACID + PHTHALYLSULFATHIAZOLE
  EXCLUDED  DICLOFENAC SODIUM + PHTHALYLSULFATHIAZOLE
  EXCLUDED  ADALIMUMAB + PHTHALYLSULFATHIAZOLE

APPLYING FILTERS

Original novel signals:     69
Removed (same-drug):        0
Removed (known missed):     0
Removed (manual exclusion): 8
Remaining novel signals:    61
  - Clean (ready for lit search):  61
  - Flagged for review:            0

 Saved: FAERS_DDI_NOVEL_CLEANED.csv (61 signals)
 Saved: FAERS_DDI_KNOWN_

### 7.10 Final DDI Analysis Summary

End-to-end summary of the thesis DDI pipeline: from 7.8M raw drug records across Q1–Q4 2025, through deduplication, RxNorm standardization, ROR calculation, signal filtering, and DrugBank validation.

In [84]:
# =============================================================================
# 7.10 SUMMARY
# =============================================================================

print("=" * 70)
print("FAERS DDI ANALYSIS - FINAL SUMMARY")
print("=" * 70)

print("\n DATA PIPELINE")
print("-" * 40)
print("   Quarters analyzed: Q1-Q4 2025")
print("   Total patients: 1,617,444")
print("   Drug pairs analyzed: 141,097")
print("   RxNorm standardization: 91.9% mapped")

print("\n SIGNAL DETECTION")
print("-" * 40)
print("   Total DDI signals: 19,741 (14% of pairs)")
print("   High-confidence signals: 1,366")
print("   Priority signals for validation: 300")

print("\n DRUGBANK VALIDATION")
print("-" * 40)
print("   Known DDIs confirmed: 44 (14.7%)")
print("   Potentially novel signals: 108 (36%)")
print("   Method validation: SUCCESSFUL")

print("\n TOP NOVEL SIGNALS FOR INVESTIGATION")
print("-" * 40)
novel = pd.read_csv("../data/validated/FAERS_DDI_NOVEL.csv")
top_novel = novel.nlargest(10, 'N_EXPOSED')[['PAIR', 'N_EXPOSED', 'ROR']]
for _, row in top_novel.iterrows():
    print(f"  • {row['PAIR']}: ROR={row['ROR']:.1f}, N={row['N_EXPOSED']}")

print("\n FILES GENERATED")
print("-" * 40)
print("  • FAERS_DDI_ROR_ALL.csv - All 141,097 pairs")
print("  • FAERS_DDI_SIGNALS.csv - 19,741 flagged signals")
print("  • FAERS_DDI_PRIORITY_LIST.csv - 300 priority signals")
print("  • FAERS_DDI_VALIDATED.csv - Validation results")
print("  • FAERS_DDI_KNOWN.csv - 44 confirmed DDIs")
print("  • FAERS_DDI_NOVEL.csv - 108 novel candidates")
print("  • DRUGBANK_DDI_REFERENCE.csv - 1.4M DrugBank DDIs")

FAERS DDI ANALYSIS - FINAL SUMMARY

 DATA PIPELINE
----------------------------------------
   Quarters analyzed: Q1-Q4 2025
   Total patients: 1,617,444
   Drug pairs analyzed: 141,097
   RxNorm standardization: 91.9% mapped

 SIGNAL DETECTION
----------------------------------------
   Total DDI signals: 19,741 (14% of pairs)
   High-confidence signals: 1,366
   Priority signals for validation: 300

 DRUGBANK VALIDATION
----------------------------------------
   Known DDIs confirmed: 44 (14.7%)
   Potentially novel signals: 108 (36%)
   Method validation: SUCCESSFUL

 TOP NOVEL SIGNALS FOR INVESTIGATION
----------------------------------------
  • METHOTREXATE + OXYCODONE: ROR=7.3, N=463
  • DICLOFENAC + RITUXIMAB: ROR=8.3, N=450
  • NAPROXEN + RITUXIMAB: ROR=6.4, N=358
  • CETIRIZINE HYDROCHLORIDE + DESOXIMETASONE: ROR=8.7, N=348
  • CETIRIZINE + RITUXIMAB: ROR=8.4, N=319
  • FOLIC ACID + OXYCODONE HYDROCHLORIDE: ROR=6.8, N=313
  • METHOTREXATE + PSEUDOEPHEDRINE HYDROCHLORIDE: ROR=